In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/311_Service_Requests.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Data Prep

In [3]:
"""
# load data
use_cols = [
    "Created Date",
    "Closed Date",
    "Agency",
    "Problem (formerly Complaint Type)",
    "Location Type",
    "Incident Zip",
    "Police Precinct",
    "Borough"
]

df = pd.read_csv(path, usecols=use_cols)
df = df.dropna()

# filter df to top 10
top10 = df['Problem (formerly Complaint Type)'].value_counts().head(10)
print(top10)

df = df[df['Problem (formerly Complaint Type)'].isin(top10.index)]
print(df.info())

df.head()

save_path = "/content/drive/MyDrive/cleaned_311_data.csv"
df.to_csv(save_path, index=False)
"""

/tmp/ipykernel_18570/2785947356.py:13: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, usecols=use_cols)


Problem (formerly Complaint Type)
Illegal Parking                        2602070
Noise - Residential                    2349228
HEAT/HOT WATER                         1595371
Noise - Street/Sidewalk                1039655
Blocked Driveway                        965936
Request Large Bulky Item Collection     630089
UNSANITARY CONDITION                    625264
Noise - Vehicle                         377897
Noise - Commercial                      369559
PLUMBING                                360285
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 10915354 entries, 13 to 20741003
Data columns (total 8 columns):
 #   Column                             Dtype 
---  ------                             ----- 
 0   Created Date                       object
 1   Closed Date                        object
 2   Agency                             object
 3   Problem (formerly Complaint Type)  object
 4   Location Type                      object
 5   Incident Zip              

In [4]:
save_path = "/content/drive/MyDrive/cleaned_311_data.csv"
df = pd.read_csv(save_path)

df.info()

In [5]:
print(df['Created Date'].min())
print(df['Created Date'].max())

df = df.rename(columns={
    'Problem (formerly Complaint Type)': 'complaint_type',
    'Created Date': 'created_date',
    'Closed Date': 'closed_date'
})

df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])

df['complaint_type'] = df['complaint_type'].str.strip().str.lower()
print(df['complaint_type'].value_counts())

01/01/2020 01:00:28 AM
12/31/2025 12:59:58 AM
complaint_type
illegal parking                        2602070
noise - residential                    2349228
heat/hot water                         1595371
noise - street/sidewalk                1039655
blocked driveway                        965936
request large bulky item collection     630089
unsanitary condition                    625264
noise - vehicle                         377897
noise - commercial                      369559
plumbing                                360285
Name: count, dtype: int64


In [6]:
# calculate resolution hours
df['resolution_hours'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600
print(f"Mean Resolution Time: {df['resolution_hours'].mean()}")

In [8]:
# break created data up into hour, day of week, and month
df['hour'] = df['created_date'].dt.hour
df['day_of_week'] = df['created_date'].dt.dayofweek
df['month'] = df['created_date'].dt.month
df['Police Precinct'] = df['Police Precinct'].str.extract(r'(\d+)').astype(float)
df.head()

,created_date,closed_date,Agency,complaint_type,Location Type,Incident Zip,Police Precinct,Borough,resolution_hours,hour,day_of_week,month
0,2026-04-06 01:55:10,2026-04-06 02:00:28,NYPD,noise - street/sidewalk,Street/Sidewalk,11233.0,81.0,BROOKLYN,0.088333,1,0,4
1,2026-04-06 01:43:15,2026-04-06 02:02:16,NYPD,noise - residential,Residential Building/House,11221.0,83.0,BROOKLYN,0.316944,1,0,4
2,2026-04-06 01:40:47,2026-04-06 02:02:22,NYPD,illegal parking,Street/Sidewalk,11694.0,100.0,QUEENS,0.359722,1,0,4
3,2026-04-06 01:39:50,2026-04-06 01:48:24,NYPD,noise - street/sidewalk,Street/Sidewalk,11233.0,81.0,BROOKLYN,0.142778,1,0,4
4,2026-04-06 01:39:00,2026-04-06 01:59:24,NYPD,illegal parking,Street/Sidewalk,11694.0,100.0,QUEENS,0.340000,1,0,4


In [9]:
# encode categorial columns
cat_cols = [
    "Agency",
    "complaint_type",
    "Location Type",
    "Incident Zip",
    "Police Precinct",
    "Borough"
]

df = pd.get_dummies(df, columns=cat_cols)
df.head()


,created_date,closed_date,resolution_hours,hour,day_of_week,month,Agency_DSNY,Agency_HPD,Agency_NYPD,complaint_type_blocked driveway,...,Police Precinct_120.0,Police Precinct_121.0,Police Precinct_122.0,Police Precinct_123.0,Borough_BRONX,Borough_BROOKLYN,Borough_MANHATTAN,Borough_QUEENS,Borough_STATEN ISLAND,Borough_Unspecified
0,2026-04-06 01:55:10,2026-04-06 02:00:28,0.088333,1,0,4,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
1,2026-04-06 01:43:15,2026-04-06 02:02:16,0.316944,1,0,4,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
2,2026-04-06 01:40:47,2026-04-06 02:02:22,0.359722,1,0,4,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
3,2026-04-06 01:39:50,2026-04-06 01:48:24,0.142778,1,0,4,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
4,2026-04-06 01:39:00,2026-04-06 01:59:24,0.340000,1,0,4,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
